# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanidhyajpg-design/flyrank/blob/main/work/notebooks/w01_research_question.ipynb)

This notebook frames a seven-week investigation before any model is trained. The starter CSV is a small, anonymized, cross-sectional snapshot. Findings are descriptive and provisional.

## 1. My lane (or freestyle) and why

**Selected lane: CTR / Engagement Opportunity Scoring (predefined core lane).**

I want to investigate how to prioritize content pages that receive meaningful Google Search impressions but have relatively low click-through rates (CTR) compared with pages in similar average-position ranges. This lane fits the starter data because it contains impressions, clicks, CTR, average position, content age, content type, search intent, and engagement measurements. The practical output would be a ranked review queue for an SEO editor, not an automatic instruction to rewrite pages. I am choosing this lane provisionally because it connects measurable search performance to a concrete editorial decision, and the next seven weeks can test whether a position- and volume-aware score is more useful than a simple rule.

## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with enough search impressions to make CTR reasonably interpretable, which pages appear to have the strongest CTR-improvement opportunities after accounting for average search position, and how should an SEO team prioritize them for human review?

- **Decision to improve:** Which content page should an SEO editor investigate first?
- **Who acts and what they do:** An SEO editor reviews the highest-ranked pages, checks search intent and page context, and decides whether a title, meta description, snippet alignment, or page content deserves manual review. The score does not automatically change a page.
- **Unit of analysis (grain):** One content page (`content_id`) per row in the starter dataset. The CSV has one row per pseudonymized content item, with trailing-90-day measurements.
- **Output:** A page-level opportunity score, a ranked shortlist, and human-readable reason codes. A provisional evaluation metric is Precision@50; the final K should match the review team's actual capacity.
- **Cost of a wrong call:** A false positive wastes editor time and may lead to an unnecessary change that does not help. A false negative can leave a worthwhile page unreviewed. Recommendations therefore require human review and should include evidence and uncertainty.
- **Why data/ML may help:** CTR is affected by several interacting signals, including position, impression volume, intent, content type, and page context. A transparent position/volume-adjusted rule is the baseline. ML is justified only if it finds stable, useful combinations of signals and improves a decision-matched metric on an honest validation split. If it does not beat the rule, the simpler rule is preferable.

## 3. Quick look at the data (2–3 real numbers)

The repository's starter CSV contains **30,000 pages across 32 pseudonymized clients**. In a first-pass screen, **16,726 pages** have at least 500 impressions in the trailing 90 days. Of the **12,023** pages that have at least 500 impressions and a recorded average position from 1 to 20, **9,759** have CTR below 0.5% (the CTR columns are expressed as percentage points, so `0.5` means 0.5%, not 50%). This is a candidate pool for investigation, not proof that every flagged page can improve.

The 0.5% threshold and 500-impression floor are provisional screening choices. Later work must compare pages within position bands, test alternative volume floors and thresholds, and check whether the resulting shortlist is useful. Rows with `avg_position == 0` mean no position data and should not be treated as rank zero.

In [ ]:
from pathlib import Path
import pandas as pd

# Prefer the repository copy; use the public anonymized raw file in Colab if needed.
local_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
local_path = next((path for path in local_candidates if path.exists()), None)
data_url = "https://raw.githubusercontent.com/sanidhyajpg-design/flyrank/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(local_path if local_path is not None else data_url)

required = {"content_id", "client_id", "impressions_90d", "avg_position", "ctr"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Starter dataset is missing required columns: {sorted(missing)}")

pages = len(df)
clients = df["client_id"].nunique()
visible_pages = df.loc[df["impressions_90d"] >= 500]
eligible = df.loc[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
]
low_ctr_candidates = eligible.loc[eligible["ctr"] < 0.5]

print(f"Starter dataset: {pages:,} pages across {clients:,} pseudonymized clients")
print(f"Pages with at least 500 impressions: {len(visible_pages):,}")
print(f"Pages with at least 500 impressions and average position 1–20: {len(eligible):,}")
print(f"Among those eligible pages, CTR below 0.5%: {len(low_ctr_candidates):,} ({len(low_ctr_candidates) / len(eligible):.1%})")
print("Note: avg_position == 0 is treated as missing position data, not rank zero.")

## 4. Careful words: what I can and can't claim

**What I can say:** The starter snapshot contains a substantial group of pages that meet the provisional impression and position screen while showing CTR below the provisional 0.5% threshold. This supports investigating a position-aware review queue. The measurements can describe observed associations and help prioritize human attention.

**What I cannot say:** I cannot claim that a low CTR was caused by a poor title, that changing a title will increase clicks, or that this analysis reveals Google's ranking algorithm. The starter data is a trailing-90-day snapshot, not a randomized experiment or a clean future-outcome study. The screen is not a validated prediction, and a low CTR does not by itself prove an opportunity. Position, query mix, device, brand demand, and other factors may matter. Any later model must avoid leakage, compare against a transparent baseline, and be validated on data separated by client or time as appropriate.

**Provisional success test:** Can a position- and volume-aware ranking produce a more useful top-50 review list than a simple, explainable rule, using a leakage-safe evaluation and human-readable reason codes? If not, retain the simpler method.

## Self-check

- [x] I selected one of the four predefined lanes.
- [x] I named the decision, the person taking action, and the action itself.
- [x] I defined the unit of analysis and the intended output.
- [x] I described the cost of false positives and false negatives.
- [x] I included real starter-dataset counts and code that recomputes them.
- [x] I explained why ML must be compared with a simple baseline.
- [x] I used careful, non-causal language and stated what I cannot claim.
- [ ] Run every cell from top to bottom, confirm the printed counts, save the executed notebook, and commit it to this repository.

**Submission:** Submit `https://github.com/sanidhyajpg-design/flyrank` on the Week 1 assignment card after the executed notebook is committed.